In [0]:
row = spark.sql("""
  SELECT DATEDIFF(
         CURRENT_DATE(),
         COALESCE(MIN(source_file_date), CURRENT_DATE())
         ) + 1 AS max_iterations
    FROM tibia_analytics.silver.worlds
""").collect()[0]

max_iterations = row["max_iterations"]

for _ in range(max_iterations):
    result = spark.sql("""
               SELECT MIN(source_file_date) AS snapshot_date
                 FROM tibia_analytics.silver.worlds
                WHERE source_file_date > (
                        SELECT COALESCE(MAX(last_seen_date), DATE '1997-01-07')
                          FROM tibia_analytics.gold.worlds
                      )
             """).collect()[0]["snapshot_date"]

    if result is None:
        break

    dbutils.notebook.run("load_table", 0)

else:
  raise RuntimeError(f"Loop exhausted after {max_iterations} iterations without NULL.")